
## Bone Fractures Detection



## Step # 01 Install the Ultralytics Package

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip install -U albumentations

In [ ]:
# Install necessary libraries
!pip install -q ultralytics supervision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.5 MB/s eta 0:00:00


In [ ]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.119 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 41.0/107.7 GB disk)


In [ ]:
import torch
torch.cuda.empty_cache()
print(torch.cuda.is_available())

False


In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image

##  Train YOLOv11 Model

In [ ]:
!unzip -q "/content/archive.zip" -d "/content/Bone_Fractures_Detection"


In [ ]:
!mkdir -p /content/Bone_Fractures_Detection/valid/images
!mkdir -p /content/Bone_Fractures_Detection/valid/labels

In [ ]:
!yolo task=detect \
  mode=train \
  model=yolo11x.yaml \
  data="/content/Bone_Fractures_Detection/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection/data.yaml" \
  epochs=5 \
  imgsz=416 \
  batch=8 \
  device=0 \
  patience=3

## Visualization of Labels

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import pathlib
import math

In [ ]:
# Path setup
train_images_dir = pathlib.Path("/content/Bone_Fractures_Detection/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection/train/images")
train_labels_dir = pathlib.Path("/content/Bone_Fractures_Detection/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection/train/labels")
train_images = list(train_images_dir.glob("*.jpg"))

# Limit number of images
max_images = 9
train_images = train_images[:max_images]

# Class names
class_names = ['Comminuted', 'Greenstick', 'Healthy', 'Linear', 'Oblique Displaced',
               'Oblique', 'Segmental', 'Spiral', 'Transverse Displaced', 'Transverse']

n = len(train_images)
print(f"Visualizing {n} training images")

if True:
    cols = 3
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
    axes = axes.flatten()

    for i in range(n):
        img = Image.open(train_images[i]).resize((320, 320))
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(train_images[i].name, fontsize=10)

        label_path = train_labels_dir / f"{train_images[i].stem}.txt"
        img_width, img_height = img.size

        # Read Labels and DRAW Bounding Boxes
        if label_path.exists():
            with open(label_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                values = line.strip().split()
                class_id = int(values[0])
                x_center, y_center, width, height = map(float, values[1:])

                # Convert YOLO format to corner format
                x1 = int((x_center - width / 2) * img_width)
                y1 = int((y_center - height / 2) * img_height)
                box_width = int(width * img_width)
                box_height = int(height * img_height)

                # Drawing Bounding Box on Image
                rect = patches.Rectangle((x1, y1), box_width, box_height, linewidth=2, edgecolor="red", facecolor="none")
                axes[i].add_patch(rect)
                axes[i].text(x1, y1 - 5, class_names[class_id], color="red", fontsize=8)

    # Remove extra axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()


In [ ]:
from IPython.display import Image, display

display(Image(filename="/content/runs/detect/train/F1_curve.png", width=700))

## Best model load and prediction

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

In [ ]:
# Best model load
model = YOLO("/content/runs/detect/train/weights/best.pt")

# Prediction
results = model.predict(source="/content/image1.jpeg", save=True, conf=0.25)

In [ ]:
# Display prediction manually
display(Image(filename='/content/runs/detect/predict/image1.jpg'))

## prediction

In [ ]:
# Prediction
results = model.predict(source="/content/image2.jpeg", save=True, conf=0.25)

In [ ]:
# Display prediction manually
display(Image(filename='/content/runs/detect/predict2/image2.jpg'))

In [ ]:
# Prediction
results = model.predict(source="/content/img.rf.a0d6ff58486861b6e6971a8cef086d3c.jpg", save=True, conf=0.25)

In [ ]:
# Display prediction manually
display(Image(filename='/content/runs/detect/predict2/img.rf.a0d6ff58486861b6e6971a8cef086d3c.jpg'))